In [1]:
%%capture
!pip install -U pip

# Fix Pillow/PIL mismatch trên Kaggle
!pip uninstall -y pillow PIL
!pip install --no-cache-dir --force-reinstall "pillow==10.4.0"

# Clone PE repo
!rm -rf /kaggle/working/perception_models
!git clone --depth 1 https://github.com/facebookresearch/perception_models.git /kaggle/working/perception_models

# Install PE repo + deps
!pip install -e /kaggle/working/perception_models
!pip install --no-cache-dir "timm==1.0.12" einops

# API server
!pip install -U fastapi uvicorn nest_asyncio pydantic requests

In [2]:
import sys
sys.path.append("/kaggle/working/perception_models")

import core.vision_encoder.pe as pe
import core.vision_encoder.transforms as pe_transforms

print("Import OK")

Import OK


In [3]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared
!chmod +x /kaggle/working/cloudflared

In [4]:
import sys, os, time, threading, subprocess, re
from typing import List, Optional

import torch
import numpy as np

sys.path.append("/kaggle/working/perception_models")

import core.vision_encoder.pe as pe
import core.vision_encoder.transforms as pe_transforms

PE_DEVICE = "cuda:0"

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print("Loading PE-Core-G14-448 on", PE_DEVICE)

pe_model = pe.CLIP.from_config("PE-Core-G14-448", pretrained=True)
pe_model = pe_model.to(PE_DEVICE).eval()

pe_tokenizer = pe_transforms.get_text_tokenizer(pe_model.context_length)

print("PE loaded")
print("context_length:", pe_model.context_length)
print("image_size:", pe_model.image_size)

Loading PE-Core-G14-448 on cuda:0


PE-Core-G14-448.pt:   0%|          | 0.00/9.68G [00:00<?, ?B/s]

Missing keys for loading model: []
Unexpected keys for loading model: []
PE loaded
context_length: 72
image_size: 448


In [5]:
@torch.inference_mode()
def encode_pe_text(texts: List[str]) -> np.ndarray:
    """
    Return normalized PE-Core-G14 text embeddings.
    Expected shape: [N, 1280], dtype float32.
    """
    if isinstance(texts, str):
        texts = [texts]

    tokens = pe_tokenizer(texts).to(PE_DEVICE)

    with torch.autocast("cuda", dtype=torch.float16):
        if hasattr(pe_model, "encode_text"):
            feats = pe_model.encode_text(tokens)
        elif hasattr(pe_model, "get_text_features"):
            feats = pe_model.get_text_features(tokens)
        else:
            raise RuntimeError("Cannot find encode_text/get_text_features on PE model")

        feats = torch.nn.functional.normalize(feats.float(), dim=-1)

    return feats.detach().cpu().numpy().astype("float32")


# Sanity test
vec = encode_pe_text(["a flooded street with people pushing motorbikes"])
print("shape:", vec.shape)
print("dtype:", vec.dtype)
print("norm:", np.linalg.norm(vec[0]))

shape: (1, 1280)
dtype: float32
norm: 1.0


In [6]:
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel, Field
import nest_asyncio
import uvicorn

# Đổi token này nếu muốn bảo vệ endpoint.
# Backend sẽ gửi Authorization: Bearer <token>
SERVICE_TOKEN = os.environ.get("AIC26_SERVICE_TOKEN", "change-me-dev-token")

app = FastAPI(title="AIC26 PE-Core-G14 Text Encoder")

pe_lock = threading.Lock()

def check_auth(authorization: Optional[str]):
    # Nếu để default thì bỏ qua auth cho dev
    if SERVICE_TOKEN == "change-me-dev-token":
        return
    if authorization != f"Bearer {SERVICE_TOKEN}":
        raise HTTPException(status_code=401, detail="Unauthorized")

class EncodeTextRequest(BaseModel):
    texts: List[str] = Field(..., min_length=1)

@app.get("/health")
def health():
    return {
        "ok": True,
        "model": "facebook/PE-Core-G14-448",
        "device": PE_DEVICE,
        "cuda_devices": torch.cuda.device_count(),
        "context_length": getattr(pe_model, "context_length", None),
        "image_size": getattr(pe_model, "image_size", None),
    }

@app.post("/encode-text")
def api_encode_text(req: EncodeTextRequest, authorization: Optional[str] = Header(None)):
    check_auth(authorization)

    t0 = time.time()

    with pe_lock:
        vectors = encode_pe_text(req.texts)

    return {
        "model": "facebook/PE-Core-G14-448",
        "dim": int(vectors.shape[1]),
        "latency_ms": round((time.time() - t0) * 1000, 2),
        "vectors": vectors.tolist(),
    }

nest_asyncio.apply()

def run_api():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()

print("FastAPI running on http://localhost:8000")

FastAPI running on http://localhost:8000


INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [13]:
import requests

BASE = "http://127.0.0.1:8000"

print(requests.get(f"{BASE}/health").json())

r = requests.post(
    f"{BASE}/encode-text",
    json={
        "texts": [
            "a flooded street with people in raincoats pushing motorbikes"
        ]
    },
    timeout=60,
)

print("status:", r.status_code)
j = r.json()
print("model:", j["model"])
print("dim:", j["dim"])
print("latency_ms:", j["latency_ms"])
print("vector_len:", len(j["vectors"][0]))
print("first_5:", j["vectors"][0][:5])

INFO:     127.0.0.1:53478 - "GET /health HTTP/1.1" 200 OK
{'ok': True, 'model': 'facebook/PE-Core-G14-448', 'device': 'cuda:0', 'cuda_devices': 2, 'context_length': 72, 'image_size': 448}
INFO:     127.0.0.1:53494 - "POST /encode-text HTTP/1.1" 200 OK
status: 200
model: facebook/PE-Core-G14-448
dim: 1280
latency_ms: 26.19
vector_len: 1280
first_5: [-0.03414865955710411, 0.018700456246733665, -0.013506559655070305, 0.008415811695158482, -0.018979568034410477]


In [14]:
import subprocess, time, re

!pkill -f cloudflared || true

proc = subprocess.Popen(
    [
        "/kaggle/working/cloudflared",
        "tunnel",
        "--url",
        "http://localhost:8000",
        "--no-autoupdate"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None

for _ in range(80):
    line = proc.stdout.readline()
    print(line, end="")
    m = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
    if m:
        public_url = m.group(0)
        break
    time.sleep(0.5)

print("\nPUBLIC_URL =", public_url)

2026-06-07T12:25:07Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-06-07T12:25:07Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-06-07T12:25:10Z INF +--------------------------------------------------------------------------------------------+
2026-06-07T12:25:10Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-06-07T12:25:10Z INF |  https://aggregate-gorgeous-lunch-lane.trycloudflare.c

## 6. GLAP text encoder cho AUDIO vector search (bổ sung)

Trước đây thiếu cell này nên backend chỉ search audio bằng tag/caption (Elastic) → audio search có thể sai. Hai cell dưới:
1. **Cài `glap_model`** (dasheng-glap) — đúng model đã tạo `.glap.npy` (collection `aic26_audio_glap_v1`).
2. **Load GLAP + thêm endpoint `/encode-audio-text`** vào app đang chạy (cùng port 8000 / cùng cloudflared tunnel).

Chạy **sau** cell khởi động server. Backend gọi `{PE_ENCODER_URL}/encode-audio-text` (không cần biến mới), encode text query → vector 1024-d → search Milvus `aic26_audio_glap_v1` (cosine).


In [ ]:
# Cài GLAP (dasheng-glap) cho audio↔text embedding — cùng model đã tạo .glap.npy.
!pip install glap_model || pip install "git+https://github.com/xiaomi-research/dasheng-glap.git"
# QUAN TRỌNG: glap_model (qua dasheng) cần transformers < 5.0, nếu không sẽ lỗi
#   ImportError: cannot import name 'GenerationMixin' from 'transformers.generation'
# Kaggle mặc định transformers 5.0 → phải hạ về 4.53.3 (bản đã xác nhận chạy GLAP).
!pip install "transformers==4.53.3"
!python -c "import transformers; from glap_model import glap_inference; print('>>> transformers', transformers.__version__, '| glap_model OK')"
# ⚠️ SAU cell này PHẢI **Restart kernel** rồi chạy lại TỪ ĐẦU (cả PE + GLAP dùng 4.53.3).
#    Nếu PE lỗi với 4.53.3 → dùng notebook GLAP RIÊNG (glap-encoder-kaggle.ipynb) +
#    đặt GLAP_ENCODER_URL trỏ tunnel của nó (backend đã hỗ trợ tách riêng).


In [ ]:
# ===== GLAP text encoder (audio↔text) — bổ sung cho AUDIO VECTOR SEARCH =====
# Khớp ĐÚNG model đã tạo .glap.npy (collection aic26_audio_glap_v1): mispeech/GLAP,
# 1024-d, L2-normalized — cùng chuẩn hóa với encode_audio để cosine có nghĩa.
# Route được THÊM NÓNG vào app đang chạy (không cần restart server). Cùng tunnel/port 8000
# => backend gọi {PE_ENCODER_URL}/encode-audio-text (hoặc đặt GLAP_ENCODER_URL riêng).
import numpy as np, torch, threading, time
from typing import List, Optional
from fastapi import Header
from pydantic import BaseModel, Field

from glap_model import glap_inference

# CPU cho text encoder: encode query rất nhẹ; tránh lỗi
#   "Expected all tensors to be on the same device ... index is on cpu ... cuda:0"
# (glap.encode_text không tự đưa token ids lên GPU). Embedding vẫn cùng không gian .glap.npy.
GLAP_DEVICE = "cpu"
print("Loading GLAP (mispeech/GLAP) on", GLAP_DEVICE)
glap_model = glap_inference()
try:
    glap_model = glap_model.to(GLAP_DEVICE).eval()
except Exception:
    pass
glap_lock = threading.Lock()

@torch.inference_mode()
def encode_glap_text(texts: List[str]) -> np.ndarray:
    """GLAP text embedding [N,1024] float32, L2-normalized (giống .glap.npy)."""
    if isinstance(texts, str):
        texts = [texts]
    emb = glap_model.encode_text(list(texts)).float()
    emb = emb / emb.norm(dim=-1, keepdim=True).clamp_min(1e-8)
    return emb.detach().cpu().numpy().astype("float32")

# Sanity
_v = encode_glap_text(["a car horn honking in traffic", "piano music"])
print("GLAP text dim:", _v.shape, "| norm:", float(np.linalg.norm(_v[0])))

class EncodeAudioTextRequest(BaseModel):
    texts: List[str] = Field(..., min_length=1)

@app.post("/encode-audio-text")
def api_encode_audio_text(req: EncodeAudioTextRequest, authorization: Optional[str] = Header(None)):
    check_auth(authorization)
    t0 = time.time()
    with glap_lock:
        vectors = encode_glap_text(req.texts)
    return {
        "model": "mispeech/GLAP",
        "dim": int(vectors.shape[1]),
        "latency_ms": round((time.time() - t0) * 1000, 2),
        "vectors": vectors.tolist(),
    }

print(">>> /encode-audio-text sẵn sàng (route hot-added vào app đang chạy).")
print('>>> Test:  POST {PUBLIC_URL}/encode-audio-text  {"texts":["piano music"]}')
